# TimbreTrack — a simple example

Extract psychoacoustic timbre features from a `.wav` file with
`comsar.TimbreTrack`, then visualise them: the waveform is drawn in grey with the
feature tracks overlaid in colour, together with a **play button** and a cursor
that follows the audio.

> Requires `bader-comsar` (see the [main README](../README.md)) plus
> `soundfile`, and a browser that can play WAV audio.

In [ ]:
from comsar import TimbreTrack
from comsar.viz import timbre_player

## Analysis parameters

`TimbreTrack` cuts the recording into short **analysis windows** and computes
every timbre feature once per window. The windowing is specified in **time
units**, not in samples:

| Parameter | Meaning |
|---|---|
| `window_ms` | Length of one analysis window in **milliseconds**. Longer windows → finer frequency resolution, coarser time resolution. |
| `overlap` | Overlap between consecutive windows as a **fraction** of the window length (`0 < overlap < 1`). More overlap → denser, smoother feature track, longer computation. |

The **hop size** (time step between two feature values) is
`window_ms × (1 − overlap)`. With the values below — 370 ms windows and 80 %
overlap — you get one feature value every 74 ms.

**Any sample rate works.** The window and hop are converted to samples using
each file's own sample rate, so recordings with *different sample rates stay
comparable*: two files of equal duration produce exactly the same number of
analysis values, and the feature table carries the frame time in seconds
(`time_s`) as its index.

In [ ]:
# --- Analysis parameters ------------------------------------------------------
WINDOW_MS        = 370.0   # length of one analysis window in milliseconds
OVERLAP_FRACTION = 0.8     # fraction of the window that overlaps (0 ... <1)

# Works with any sample rate: windows are converted to samples per file.
tt = TimbreTrack(window_ms=WINDOW_MS, overlap=OVERLAP_FRACTION)
print(f"hop = {WINDOW_MS * (1 - OVERLAP_FRACTION):.1f} ms per feature value")

In [ ]:
# Extract psychoacoustic features from the recording. The path is relative, so
# the .wav file has to sit next to this notebook. `extract` returns a
# TrackResult; `.features` is a pandas DataFrame with one row per analysis frame.
WAV = "CHI-109_Luguhu_Hulusheng.wav"
WAV = "CAM-5_SARA PANH.wav"
result = tt.extract(WAV)
features = result.features
features.head()

## Save the results as CSV

The extracted features are saved right away, so the analysis is not lost if the
notebook is closed. The table's index is the frame time in seconds (`time_s`) —
it becomes the first CSV column automatically. Because the windowing is defined
in milliseconds, two recordings of equal duration yield CSV files with the
same number of rows, **regardless of their sample rates**.

**Excel note.** German/European Excel uses the comma as *decimal* sign and
therefore expects **semicolons** between columns. With `EXCEL_STYLE = True`
(default) the file opens cleanly in European Excel; set it to `False` for a
standard comma CSV (pandas, R, US Excel). To read the Excel-style file back:
`pd.read_csv(CSV_FILE, sep=";", decimal=",", index_col=0)`.

In [ ]:
# Save the feature table as CSV; the index already holds the time in seconds
CSV_FILE = WAV.rsplit(".", 1)[0] + "_timbre.csv"

# German/European Excel expects ';' as column separator and ',' as decimal
# sign. Set EXCEL_STYLE = False for a standard comma-separated CSV.
EXCEL_STYLE = True

if EXCEL_STYLE:
    features.to_csv(CSV_FILE, sep=";", decimal=",")
else:
    features.to_csv(CSV_FILE)
print(f"saved {len(features)} frames x {len(features.columns)} features to {CSV_FILE}")

## Visualisation

`comsar.viz.timbre_player` draws the waveform in light grey with each feature
normalised to `[0, 1]` and overlaid in its own colour. Press **Play** to listen
— a vertical cursor follows the playback position, and clicking the plot seeks
to that point.

To keep the plot readable, only the **first two features are shown initially**;
**click a legend entry** to show or hide a feature (hidden entries are grey).
Use `visible=None` to start with all features shown, e.g.
`timbre_player(WAV, features, visible=None)`.

The player is a self-contained HTML/JavaScript widget (audio embedded as a data
URI), so it also works after the notebook is exported to HTML.

Because `TimbreTrack` also computes the wavelet roughness, the result carries the detected **partial frequencies** in `result.partials`. Passing them to the player adds a bottom panel showing those frequencies over time (grey level = amplitude); the clickable **partials** legend entry hides/shows it.


In [ ]:
# Pass `partials=result.partials` so the player also shows the partial-gram
# panel (partial frequencies over time, grey = amplitude). Click the "partials"
# legend entry to hide/show that panel.
timbre_player(WAV, features, partials=result.partials)